In [1]:
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
file_path = '/content/drive/MyDrive/ML Project/mta_hourly_with_lags.csv'

In [3]:
mta_df = pd.read_csv(file_path)

In [4]:
mta_df['transit_timestamp'] = pd.to_datetime(mta_df['transit_timestamp'])
print(f"Loaded {len(mta_df)} rows from Drive.")
mta_df.head()

Loaded 105840 rows from Drive.


,hub_name,transit_timestamp,ridership,hour,day_of_week,hour_sin,hour_cos,day_sin,day_cos,lag_1h,lag_24h
0,Fulton Center,2022-05-16 09:00:00,1471,9,0,7.071068e-01,-0.707107,0.0,1.0,1662.0,675.0
1,Fulton Center,2022-05-16 10:00:00,1506,10,0,5.000000e-01,-0.866025,0.0,1.0,1471.0,1025.0
2,Fulton Center,2022-05-16 11:00:00,1401,11,0,2.588190e-01,-0.965926,0.0,1.0,1506.0,1418.0
3,Fulton Center,2022-05-16 12:00:00,1654,12,0,1.224647e-16,-1.000000,0.0,1.0,1401.0,1750.0
4,Fulton Center,2022-05-16 13:00:00,2065,13,0,-2.588190e-01,-0.965926,0.0,1.0,1654.0,1687.0


The coordinate is located on the downtown Manhattan and date parameter include only the range of our mta dataset.

In [5]:
import requests
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 40.7128,
    "longitude": -74.0060,
    "start_date": "2022-05-15",
    "end_date": "2024-05-20",
    "hourly": ["temperature_2m", "precipitation", "wind_speed_10m"],
    "timezone": "America/New_York"
}
response = requests.get(url, params=params)
weather_json = response.json()

In [6]:
weather_df = pd.DataFrame({
    "weather_timestamp": pd.to_datetime(weather_json["hourly"]["time"]),
    "temperature_2m": weather_json["hourly"]["temperature_2m"],
    "precipitation": weather_json["hourly"]["precipitation"],
    "wind_speed_10m": weather_json["hourly"]["wind_speed_10m"]
})

In [7]:
weather_df['weather_timestamp'] = weather_df['weather_timestamp'].dt.tz_localize(None)
print(f"Fetched {len(weather_df)} hours of NYC weather data.")
weather_df.head()

Fetched 17688 hours of NYC weather data.


,weather_timestamp,temperature_2m,precipitation,wind_speed_10m
0,2022-05-15 00:00:00,14.5,0.0,7.8
1,2022-05-15 01:00:00,14.5,0.0,4.9
2,2022-05-15 02:00:00,14.7,0.0,5.1
3,2022-05-15 03:00:00,14.9,0.0,5.5
4,2022-05-15 04:00:00,15.1,0.0,3.3


temperature_2m: The air temperature measured exactly 2 meters above the ground(in Celsius)

precipitation: The total amount of rain or snow that fell during that hour (in millimeters).

wind_speed_10m: The wind speed measured 10 meters above ground level (in kilometers per hour).

In [8]:
weather_df[['temperature_2m', 'precipitation', 'wind_speed_10m']] = weather_df[['temperature_2m', 'precipitation', 'wind_speed_10m']].ffill()
final_merged_df = pd.merge(
    mta_df,
    weather_df,
    left_on='transit_timestamp',
    right_on='weather_timestamp',
    how='left'
)
final_merged_df = final_merged_df.drop(columns=['weather_timestamp'])
print(final_merged_df[['transit_timestamp', 'hub_name', 'ridership', 'temperature_2m', 'precipitation']].head())

    transit_timestamp       hub_name  ridership  temperature_2m  precipitation
0 2022-05-16 09:00:00  Fulton Center       1471            21.2            0.0
1 2022-05-16 10:00:00  Fulton Center       1506            21.9            0.0
2 2022-05-16 11:00:00  Fulton Center       1401            23.3            0.0
3 2022-05-16 12:00:00  Fulton Center       1654            22.0            0.8
4 2022-05-16 13:00:00  Fulton Center       2065            18.5            1.1


In [9]:
!pip install holidays

In [10]:
import holidays
ny_holidays = holidays.US(state='NY', years=[2022, 2023, 2024])
final_merged_df['date_only'] = final_merged_df['transit_timestamp'].dt.date
final_merged_df['is_holiday'] = final_merged_df['date_only'].apply(lambda x: 1 if x in ny_holidays else 0)
final_merged_df['holiday_name'] = final_merged_df['date_only'].apply(lambda x: ny_holidays.get(x, 'None'))
final_merged_df = final_merged_df.drop(columns=['date_only'])
print(f"Total holiday hours tagged: {final_merged_df['is_holiday'].sum()}")
final_merged_df[final_merged_df['is_holiday'] == 1].head()

Total holiday hours tagged: 4752


,hub_name,transit_timestamp,ridership,hour,day_of_week,hour_sin,hour_cos,day_sin,day_cos,lag_1h,lag_24h,temperature_2m,precipitation,wind_speed_10m,is_holiday,holiday_name
327,Fulton Center,2022-05-30 00:00:00,257,0,0,0.000000,1.000000,0.0,1.0,593.0,404.0,17.2,0.0,8.7,1,Memorial Day
328,Fulton Center,2022-05-30 01:00:00,139,1,0,0.258819,0.965926,0.0,1.0,257.0,131.0,16.1,0.0,6.6,1,Memorial Day
329,Fulton Center,2022-05-30 02:00:00,48,2,0,0.500000,0.866025,0.0,1.0,139.0,71.0,15.4,0.0,7.6,1,Memorial Day
330,Fulton Center,2022-05-30 03:00:00,34,3,0,0.707107,0.707107,0.0,1.0,48.0,54.0,15.4,0.0,6.6,1,Memorial Day
331,Fulton Center,2022-05-30 04:00:00,46,4,0,0.866025,0.500000,0.0,1.0,34.0,44.0,15.1,0.0,7.2,1,Memorial Day


In [11]:
final_ml_matrix = pd.get_dummies(
    final_merged_df,
    columns=['hub_name', 'holiday_name'],
    drop_first=False
)
bool_cols = final_ml_matrix.select_dtypes(include=['bool']).columns
final_ml_matrix[bool_cols] = final_ml_matrix[bool_cols].astype(int)

In [12]:
# Save the final matrix to your shared Drive folder
save_path = '/content/drive/MyDrive/ML Project/final_mta_ml_matrix.csv'
final_ml_matrix.to_csv(save_path, index=False)
print(f"Ready for modeling! Saved {len(final_ml_matrix)} rows to {save_path}")

Ready for modeling! Saved 105840 rows to /content/drive/MyDrive/ML Project/final_mta_ml_matrix.csv
